# RuSearchRank Phase 3 — frozen Kaggle GPU production runner

This notebook runs the unchanged Phase 3 ML protocol for `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` at pinned model/tokenizer revision `1427fd652930e4ba29e8149678df786c240d8825`.

Attach one private Kaggle Dataset containing the official Phase 1 and Phase 2 ZIP files, choose a GPU accelerator (T4 x2 is supported, but the pipeline intentionally uses only `cuda:0`), and run cells individually from top to bottom. Setup and data-staging cells 2–9 are CPU work even though the GPU accelerator must already be enabled for preflight; GPU execution starts at S0 in cell 10.

Do **not** use **Save & Run All** and do not run C1/A1/A2/B1 in parallel. If training is interrupted, reconnect to the same `/kaggle/working` state and rerun the same run cell with `RESUME_TRAINING = True`; the runner adds `--resume` only when that run's manifest exists and never combines it with `--overwrite`. Valid checkpoints, the venv, pinned tokenizer, and Phase 3 artifacts are preserved. After packaging and downloading the ZIP files from `phase3-final`, manually stop the GPU session.

The dev boundary remains sealed until `select-checkpoint`: pre-selection cells do not load dev qrels, dev annotations, or fine-tuned dev scores.


In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')
if platform.system() != 'Linux':
    raise RuntimeError('Phase 3 Kaggle production protocol requires Linux')
for required in (KAGGLE_WORKING, KAGGLE_INPUT):
    if not required.is_dir():
        raise RuntimeError(f'missing required Kaggle path: {required}')

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('nvidia-smi failed; enable a Kaggle GPU accelerator')
print(gpu.stdout)
gpu_list = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=True)
print('GPU list:\n' + gpu_list.stdout)

import torch
cuda_available = torch.cuda.is_available()
print({'pytorch': torch.__version__, 'cuda_runtime': torch.version.cuda, 'cuda_available': cuda_available})
if not cuda_available:
    raise RuntimeError('CUDA is unavailable; enable a Kaggle GPU accelerator')
device = torch.device('cuda:0')
properties = torch.cuda.get_device_properties(device)
free_vram, total_vram = torch.cuda.mem_get_info(device)

meminfo = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, value = line.split(':', 1)
    meminfo[key] = int(value.strip().split()[0]) * 1024
total_ram = meminfo['MemTotal']
available_ram = meminfo['MemAvailable']
disk = shutil.disk_usage(KAGGLE_WORKING)
report = {
    'linux': platform.platform(),
    'cuda:0_name': torch.cuda.get_device_name(device),
    'compute_capability': f'{properties.major}.{properties.minor}',
    'vram_total_bytes': total_vram,
    'vram_free_bytes': free_vram,
    'cpu_ram_total_bytes': total_ram,
    'cpu_ram_available_bytes': available_ram,
    'disk_free_bytes': disk.free,
}
print(report)
MIN_TOTAL_RAM_GIB = 12
MIN_FREE_DISK_GIB = 25
if total_ram < MIN_TOTAL_RAM_GIB * 1024**3:
    raise RuntimeError('at least 12 GiB total CPU RAM is required')
if disk.free < MIN_FREE_DISK_GIB * 1024**3:
    raise RuntimeError('at least 25 GiB free disk is required')
if available_ram < 4 * 1024**3:
    print('WARNING: available CPU RAM is below 4 GiB; total RAM passed the hard check')


In [ ]:
import hashlib, json

BRANCH = 'phase-3-kaggle'
REPOSITORY_URL = 'https://github.com/kopanevk/ru-search-rank.git'
REPOSITORY = Path('/kaggle/working/ru-search-rank')
ALLOW_OVERWRITE_PHASE3 = False
RESUME_TRAINING = True
if ALLOW_OVERWRITE_PHASE3 and RESUME_TRAINING:
    raise RuntimeError('--resume and --overwrite must never be enabled together')

def run_checked(command, *, cwd=REPOSITORY, stream=False):
    print('$', ' '.join(map(str, command)), flush=True)
    if stream:
        result = subprocess.run(command, cwd=cwd, text=True)
    else:
        result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'command failed with return code {result.returncode}; review the complete log')
    return result

def training_options(run_id):
    if globals().get('FINETUNE_SMOKE_PASSED') is not True:
        raise RuntimeError('real smoke must pass before any finetune run')
    if ALLOW_OVERWRITE_PHASE3:
        return ['--overwrite']
    manifest = REPOSITORY / f'artifacts/models/{run_id}/run_manifest.json'
    return ['--resume'] if RESUME_TRAINING and manifest.is_file() else []

if not REPOSITORY.is_dir():
    run_checked(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, str(REPOSITORY)], cwd=KAGGLE_WORKING, stream=True)
branch = run_checked(['git', 'branch', '--show-current']).stdout.strip()
status = run_checked(['git', 'status', '--short']).stdout.strip()
if branch != BRANCH or status:
    raise RuntimeError(f'repository must be a clean {BRANCH} checkout')
run_checked(['git', 'pull', '--ff-only', 'origin', BRANCH], stream=True)


In [ ]:
import re
from datetime import datetime, timezone

TREC_EVAL_TAG = 'v9.0.8'
TREC_EVAL_COMMIT = 'd95ca64e14a47d763ae349fb65e6d8cde4141dbd'
TREC_EVAL_DIR = Path('/kaggle/working/trec_eval')
run_checked(['apt-get', 'update'], cwd=KAGGLE_WORKING, stream=True)
run_checked(['apt-get', 'install', '-y', 'openjdk-21-jdk-headless', 'build-essential'], cwd=KAGGLE_WORKING, stream=True)
if TREC_EVAL_DIR.exists():
    shutil.rmtree(TREC_EVAL_DIR)
run_checked(['git', 'clone', '--depth', '1', '--branch', TREC_EVAL_TAG, 'https://github.com/usnistgov/trec_eval.git', str(TREC_EVAL_DIR)], cwd=KAGGLE_WORKING)
head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=TREC_EVAL_DIR).stdout.strip()
tagged = run_checked(['git', 'rev-list', '-n', '1', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR).stdout.strip()
if head != tagged or head != TREC_EVAL_COMMIT:
    raise RuntimeError(f'unexpected trec_eval commit {head}')
run_checked(['git', 'diff', '--quiet', 'HEAD'], cwd=TREC_EVAL_DIR)
run_checked(['git', 'diff', '--cached', '--quiet'], cwd=TREC_EVAL_DIR)
makefile_sha256_before = hashlib.sha256((TREC_EVAL_DIR / 'Makefile').read_bytes()).hexdigest()
jobs = os.cpu_count() or 2
build_command = f'make -j{jobs}'
run_checked(['make', f'-j{jobs}'], cwd=TREC_EVAL_DIR, stream=True)
run_checked(['git', 'diff', '--quiet', 'HEAD'], cwd=TREC_EVAL_DIR)
run_checked(['git', 'diff', '--cached', '--quiet'], cwd=TREC_EVAL_DIR)
makefile_sha256_after = hashlib.sha256((TREC_EVAL_DIR / 'Makefile').read_bytes()).hexdigest()
if makefile_sha256_after != makefile_sha256_before:
    raise RuntimeError('upstream Makefile changed during build')
Path('/opt/bin').mkdir(parents=True, exist_ok=True)
run_checked(['install', '-m', '0755', str(TREC_EVAL_DIR / 'trec_eval'), '/opt/bin/trec_eval'], cwd=KAGGLE_WORKING)
version_probe = run_checked(['/opt/bin/trec_eval', '-v'], cwd=KAGGLE_WORKING)
if not re.search(r'\b9\.0\.7\b', version_probe.stdout or version_probe.stderr):
    raise RuntimeError('official v9.0.8 binary must report 9.0.7')
binary_sha256 = hashlib.sha256(Path('/opt/bin/trec_eval').read_bytes()).hexdigest()
compiler = run_checked(['cc', '--version'], cwd=KAGGLE_WORKING).stdout.splitlines()[0]
provenance = {
    'source_repository': 'https://github.com/usnistgov/trec_eval.git',
    'source_tag': TREC_EVAL_TAG,
    'source_commit': head,
    'source_tree_clean': True,
    'fresh_checkout': True,
    'source_path': str(TREC_EVAL_DIR),
    'makefile_sha256': makefile_sha256_after,
    'binary_path': '/opt/bin/trec_eval',
    'binary_sha256': binary_sha256,
    'binary_reported_version': '9.0.7',
    'expected_release_version': '9.0.8',
    'known_upstream_version_string_mismatch': True,
    'build_command': build_command,
    'compiler': compiler,
    'built_at': datetime.now(timezone.utc).isoformat(),
}
provenance_path = REPOSITORY / 'artifacts/work/phase2/trec_eval_build_provenance.json'
provenance_path.parent.mkdir(parents=True, exist_ok=True)
provenance_path.write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
print(json.dumps(provenance, indent=2))


In [ ]:
import venv

VENV_PATH = Path('/kaggle/working/rusearchrank-phase3-venv')
venv_python = VENV_PATH / 'bin/python'
if venv_python.is_file():
    probe = subprocess.run([str(venv_python), '-c', 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")'], capture_output=True, text=True)
    if probe.returncode != 0 or probe.stdout.strip() != '3.12':
        print('Existing venv is not a valid Python 3.12 environment; recreating it')
        shutil.rmtree(VENV_PATH)
if not venv_python.is_file():
    if sys.version_info[:2] == (3, 12):
        venv.EnvBuilder(with_pip=True).create(VENV_PATH)
    else:
        print(f'Kernel Python is {platform.python_version()}; installing Python 3.12 through uv')
        run_checked([sys.executable, '-m', 'pip', 'install', 'uv'], cwd=KAGGLE_WORKING, stream=True)
        run_checked([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', str(VENV_PATH)], cwd=KAGGLE_WORKING, stream=True)
RUN_PYTHON = str(venv_python)
run_checked([RUN_PYTHON, '-c', "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)"], cwd=REPOSITORY)
run_checked([RUN_PYTHON, '-m', 'pip', 'install', '-U', 'pip'], cwd=REPOSITORY, stream=True)
run_checked([RUN_PYTHON, '-m', 'pip', 'install', '-e', '.'], cwd=REPOSITORY, stream=True)
run_checked([RUN_PYTHON, '-c', 'import rusearchrank, torch, transformers, tokenizers, yaml; print("project imports PASS")'], cwd=REPOSITORY)


In [ ]:
config_probe = "import json, yaml; from pathlib import Path; print(json.dumps(yaml.safe_load(Path('configs/finetune.yaml').read_text(encoding='utf-8'))))"
config = json.loads(run_checked([RUN_PYTHON, '-c', config_probe], cwd=REPOSITORY).stdout)
golden = json.loads((REPOSITORY / 'tests/fixtures/pair_encoding_golden.json').read_text(encoding='utf-8'))
MODEL_ID = config['base_model']['id']
TOKENIZER_REVISION = config['base_model']['tokenizer_revision']
if MODEL_ID != golden['model_id'] or TOKENIZER_REVISION != golden['tokenizer_revision']:
    raise RuntimeError('config and golden tokenizer identity differ')
expected_hashes = golden['tokenizer_payload_sha256']
PINNED_TOKENIZER_DIR = Path('/kaggle/working/rusearchrank-pinned-tokenizer')
PINNED_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

def valid_tokenizer_file(name, digest):
    path = PINNED_TOKENIZER_DIR / name
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest

missing = [name for name, digest in expected_hashes.items() if not valid_tokenizer_file(name, digest)]
if missing:
    download_code = '''import hashlib, json, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download
repo, revision, destination, expected = __import__('sys').argv[1:]
destination = Path(destination); hashes = json.loads(expected)
for name in hashes:
    cached = Path(hf_hub_download(repo_id=repo, filename=name, revision=revision))
    target = destination / name
    shutil.copy2(cached, target)
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != hashes[name]: raise RuntimeError(f'tokenizer hash mismatch for {name}: {actual}')
'''
    run_checked([RUN_PYTHON, '-c', download_code, MODEL_ID, TOKENIZER_REVISION, str(PINNED_TOKENIZER_DIR), json.dumps(expected_hashes)], cwd=REPOSITORY, stream=True)
for name, digest in expected_hashes.items():
    if not valid_tokenizer_file(name, digest):
        raise RuntimeError(f'pinned tokenizer validation failed: {name}')
os.environ['RUSEARCHRANK_PINNED_TOKENIZER_DIR'] = str(PINNED_TOKENIZER_DIR)
offline_probe = "from transformers import AutoTokenizer; import os; p=os.environ['RUSEARCHRANK_PINNED_TOKENIZER_DIR']; t=AutoTokenizer.from_pretrained(p, local_files_only=True); print(type(t).__name__, len(t))"
run_checked([RUN_PYTHON, '-c', offline_probe], cwd=REPOSITORY)
run_checked([RUN_PYTHON, '-m', 'pytest', '-q'], cwd=REPOSITORY, stream=True)
run_checked([RUN_PYTHON, '-c', "import platform, torch, transformers, tokenizers; assert torch.cuda.is_available(); print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'tokenizers': tokenizers.__version__, 'cuda': torch.version.cuda, 'device': torch.cuda.get_device_name(0)})"], cwd=REPOSITORY)


In [ ]:
for validator in ('validate_phase1_notebook.py', 'validate_phase2_notebook.py', 'validate_phase3_notebook.py'):
    run_checked([RUN_PYTHON, f'scripts/{validator}'], cwd=REPOSITORY)


In [ ]:
import zipfile
from pathlib import PurePosixPath

PHASE12_INPUT_DIR = Path('/kaggle/working/phase12-inputs')
PHASE12_INPUT_DIR.mkdir(parents=True, exist_ok=True)
markers = {'phase1': 'candidate_cache_manifest.json', 'phase2': 'rerank_manifest.json'}
matches = {phase: [] for phase in markers}
for candidate in sorted(KAGGLE_INPUT.rglob('*.zip')):
    try:
        with zipfile.ZipFile(candidate) as archive:
            bad_member = archive.testzip()
            if bad_member is not None:
                raise RuntimeError(f'CRC failed for {candidate}: {bad_member}')
            basenames = {PurePosixPath(name).name for name in archive.namelist()}
    except zipfile.BadZipFile as exc:
        raise RuntimeError(f'invalid attached ZIP: {candidate}') from exc
    kinds = [phase for phase, marker in markers.items() if marker in basenames]
    if len(kinds) > 1:
        raise RuntimeError(f'attached ZIP contains both Phase manifests: {candidate}')
    if kinds:
        matches[kinds[0]].append(candidate)
for phase, paths in matches.items():
    if len(paths) != 1:
        raise RuntimeError(f'expected exactly one {phase} ZIP in /kaggle/input, found {len(paths)}: {paths}')

def streaming_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

local_archives = {}
for phase in ('phase1', 'phase2'):
    source = matches[phase][0]
    digest = streaming_sha256(source)
    destination = PHASE12_INPUT_DIR / f'rusearchrank_{phase}_results.zip'
    print({'phase': phase, 'input_path': str(source.resolve()), 'size_bytes': source.stat().st_size, 'sha256': digest})
    if destination.exists():
        if not destination.is_file() or streaming_sha256(destination) != digest:
            raise RuntimeError(f'refusing to replace different staged archive: {destination}')
    else:
        shutil.copy2(source, destination)
    if streaming_sha256(destination) != digest:
        raise RuntimeError(f'staged archive hash mismatch: {destination}')
    local_archives[phase] = destination
PHASE1_ZIP = local_archives['phase1']
PHASE2_ZIP = local_archives['phase2']

def restore_archive(path):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        for name in names:
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise RuntimeError(f'unsafe archive member: {name}')
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f'CRC failed for {bad_member}')
        archive.extractall(REPOSITORY)
    validated_manifests = 0
    for name in names:
        if not name.endswith('manifest.json'):
            continue
        candidate = json.loads((REPOSITORY / name).read_text(encoding='utf-8'))
        entries = candidate.get('artifacts', candidate.get('files'))
        if not isinstance(entries, list):
            continue
        for entry in entries:
            payload_path = REPOSITORY / entry['path']
            if not payload_path.is_file() or payload_path.stat().st_size != entry['size_bytes']:
                raise RuntimeError(f'restored payload size mismatch: {entry["path"]}')
            if streaming_sha256(payload_path) != entry['sha256']:
                raise RuntimeError(f'restored payload hash mismatch: {entry["path"]}')
        validated_manifests += 1
    if validated_manifests != 1:
        raise RuntimeError(f'expected one payload manifest in {path.name}')

restore_archive(PHASE1_ZIP)
restore_archive(PHASE2_ZIP)
print('Phase 1/2 artifacts restored from validated Kaggle inputs')
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'prepare-annotations', '--config', 'configs/retrieval.yaml', '--split', 'train'], cwd=REPOSITORY, stream=True)
print('Pinned train annotations materialized; evaluation inputs remain sealed')


In [ ]:
CONFIG = 'configs/finetune.yaml'
snapshot_code = "import json; from pathlib import Path; from rusearchrank.training_data import load_finetune_config, phase12_immutable_snapshot; c=load_finetune_config(Path('configs/finetune.yaml')); print(json.dumps(phase12_immutable_snapshot(c), sort_keys=True))"
def phase12_snapshot():
    return json.loads(run_checked([RUN_PYTHON, '-c', snapshot_code], cwd=REPOSITORY).stdout)
phase12_inputs = phase12_snapshot()
phase12_snapshot_path = REPOSITORY / 'artifacts/work/phase3/phase12_preselection_snapshot.json'
phase12_snapshot_path.parent.mkdir(parents=True, exist_ok=True)
phase12_snapshot_path.write_text(json.dumps(phase12_inputs, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(phase12_inputs, indent=2))
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'build-training-split', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
for regime in ('judged_only', 'weak_negatives', 'control_c1'):
    run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'build-training-pairs', '--config', CONFIG, '--regime', regime] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
verify_code = "import json; from pathlib import Path; from rusearchrank.training_data import load_finetune_config, verify_phase12_immutable; c=load_finetune_config(Path('configs/finetune.yaml')); expected=json.loads(Path('artifacts/work/phase3/phase12_preselection_snapshot.json').read_text()); verify_phase12_immutable(c, expected)"
run_checked([RUN_PYTHON, '-c', verify_code], cwd=REPOSITORY)
pairs_manifest = json.loads((REPOSITORY / 'reports/audit/pairs_manifest.json').read_text(encoding='utf-8'))
for regime in ('judged_only', 'weak_negatives', 'control_c1'):
    section = pairs_manifest['regimes'][regime]
    print(json.dumps({key: section[key] for key in ('regime_id', 'usable_query_count', 'judged_pairs_before_cap', 'judged_pairs_after_cap', 'weak_pairs_before_cap', 'weak_pairs_after_cap', 'leakage_audit', 'population_disclosure', 'weight_disclosure', 'heuristic_disclosure')}, ensure_ascii=False, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'validate-checkpoint', '--config', CONFIG, '--checkpoint', 'base'], cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'smoke-finetune', '--config', CONFIG, '--limit-pairs', '64'], cwd=REPOSITORY, stream=True)
smoke = json.loads((REPOSITORY / 'reports/audit/finetune_smoke.json').read_text())
if smoke.get('status') != 'PASS' or smoke.get('real_model_forward') is not True or smoke.get('fixture_only') is not False:
    raise RuntimeError('real smoke gate did not pass')
FINETUNE_SMOKE_PASSED = True
resource = json.loads((REPOSITORY / 'reports/audit/resource_report.json').read_text())
print(json.dumps({'smoke': smoke, 'resource_report': resource, 'estimated_training_time_range_seconds': resource['estimated_training_time_range_seconds']}, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'C1'] + training_options('C1'), cwd=REPOSITORY, stream=True)
control = json.loads((REPOSITORY / 'reports/audit/control_c1.json').read_text())
if control['status'] in ('FAIL', 'BLOCKED_FOR_REVIEW'):
    raise RuntimeError(f"C1 stopped the protocol: {control['status']} — {control['reason']}")


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'A1'] + training_options('A1'), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'A2'] + training_options('A2'), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'finetune', '--config', CONFIG, '--run-id', 'B1'] + training_options('B1'), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'select-checkpoint', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
selection = json.loads((REPOSITORY / 'reports/audit/checkpoint_selection.json').read_text())
ab_report = json.loads((REPOSITORY / 'reports/metrics/validation_ab_comparison.json').read_text())
print(json.dumps({'candidates': selection['candidates'], 'best_finetuned_checkpoint': selection['best_finetuned_checkpoint'], 'production_system': selection['production_system'], 'zero_shot_won': selection['zero_shot_won'], 'exploratory_post_selection_ab': ab_report}, ensure_ascii=False, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'prepare-dev-evaluation', '--config', CONFIG], cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'score-finetuned', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'evaluate-phase3', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)
comparison = json.loads((REPOSITORY / 'reports/metrics/dev_three_way_comparison.json').read_text())
print(json.dumps({'systems': comparison['systems'], 'pipeline_status': comparison['pipeline_status'], 'ml_outcome': comparison['ml_outcome']}, ensure_ascii=False, indent=2))


In [ ]:
run_checked([RUN_PYTHON, '-m', 'rusearchrank.cli', 'package-phase3', '--config', CONFIG] + (['--overwrite'] if ALLOW_OVERWRITE_PHASE3 else []), cwd=REPOSITORY, stream=True)


In [ ]:
FINAL_OUTPUT = Path('/kaggle/working/phase3-final')
FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)
result_zip = REPOSITORY / 'artifacts/rusearchrank_phase3_results.zip'
model_zips = sorted((REPOSITORY / 'artifacts').glob('rusearchrank_phase3_model_*.zip'))
if len(model_zips) != 1:
    raise RuntimeError(f'expected exactly one Phase 3 model ZIP, found {len(model_zips)}: {model_zips}')
for source in (result_zip, model_zips[0]):
    if not source.is_file():
        raise FileNotFoundError(source)
    with zipfile.ZipFile(source) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f'CRC failed for {source}: {bad_member}')
    digest = streaming_sha256(source)
    destination = FINAL_OUTPUT / source.name
    if destination.exists():
        if not destination.is_file() or streaming_sha256(destination) != digest:
            raise RuntimeError(f'refusing to overwrite different final artifact: {destination}')
    else:
        shutil.copy2(source, destination)
    size = destination.stat().st_size
    print({'absolute_path': str(destination.resolve()), 'size_bytes': size, 'size_mib': size / 1024**2, 'sha256': digest})
print('Download both ZIP files from the Kaggle Output panel, then manually stop the GPU session.')
